In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS electronics_retailer_clg.silver;

In [0]:
from pyspark.sql.functions import col, trim, when

# READ BRONZE TABLE
df = spark.table("electronics_retailer_clg.bronze.customers")

# CLEAN COLUMN NAMES
df = df.toDF(*[c.lower().replace(" ", "_") for c in df.columns])

# TRIM SPACES
for c in df.columns:
    df = df.withColumn(c, trim(col(c)))

# REMOVE INVALID PRIMARY KEYS
df = df.filter(col("customerkey").isNotNull())

# FIX DATA TYPES
df = df.withColumn("customerkey", col("customerkey").cast("int"))

# HANDLE NULL VALUES
df = df.fillna({
    "gender": "unknown",
    "continent": "unknown"
})

# STANDARDIZE GENDER
df = df.withColumn(
    "gender",
    when(col("gender").isin("Male", "Female"), col("gender"))
    .otherwise("Unknown")
)

# REMOVE DUPLICATES
df = df.dropDuplicates(["customerkey"])


# KEEP ONLY REQUIRED COLUMNS
df = df.select(
    "customerkey",
    "gender",
    "continent"
)


display(df)
df.printSchema()


# WRITE TO SILVER
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("electronics_retailer_clg.silver.customers")

print("Customers cleaned & optimized successfully")